# Smriti × CLM-8B: can a contrastive System One model pick the right memory?

Round 4 of Smriti's decision-model trials. **CLM-8B** (Contrastive-LM, Apache-2.0:
Qwen3-8B embeddings served by vLLM + the released 75 MB projection heads, `clm-serve`)
judges Smriti's top-20 candidate memories for 500 LongMemEval questions, zero-shot, in two
ways: `choice` (the question is the state, each memory an option: CLM's native ranking
primitive) and `noul` (one yes/no relevance question per memory). Same pools, same
instructions, same metrics as the Jev and Laya rounds.

**How to run:** Runtime → Change runtime type → **L4** or **A100** GPU (a T4 is too small for
the 8B encoder) → Runtime → **Run all**. Approve the Google sign-in pop-up in step 1.
About 45–75 minutes on an L4, less on an A100; most of it is installing vLLM, downloading
16 GB of weights and encoding 20,000 texts.

**Where results go:** one zip, `smriti-colab-clm8b.zip`, in your Google Drive, shared
view-only by link so the Smriti lab can fetch it. It holds judge scores, timings and logs;
the texts are public LongMemEval turns. Re-uploaded every 5 minutes.

In [ ]:
# 1 · Clone the lab and connect the results file (one Google sign-in pop-up, then walk away)
import os
import subprocess
import sys
BRANCH = "claude/beautiful-gauss-op9dft"
if not os.path.isdir("/content/Smriti"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    "https://github.com/vn-envy/Smriti", "/content/Smriti"], check=True)
os.chdir("/content/Smriti")
COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Smriti lab at", COMMIT)
sys.path.insert(0, "/content/Smriti")
from bench.lab.colab.runner import DriveResults, run, wait_http  # noqa: E402
POOLS = "audit/2026-09-25/decision-models/pools"

name, mem = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total",
                                     "--format=csv,noheader,nounits"], text=True).strip().split(", ")
print("GPU:", name, mem, "MiB")
assert int(mem) >= 22000, f"{name} has {mem} MiB; CLM-8B needs an L4 (24 GB) or A100. Runtime → Change runtime type."
OUT, LOG = "/content/clm8b", "/content/clm8b/logs"
os.makedirs(LOG, exist_ok=True)
res = DriveResults("smriti-colab-clm8b.zip")
res.add(OUT)
res.push()

In [ ]:
# 2 · Install CLM with vLLM 0.19.1: the newest vLLM on torch 2.10 / CUDA 12.8. Colab runtimes ship
#     CUDA 12; vLLM >= 0.20 pins torch >= 2.11, built for CUDA 13 (libcudart.so.13 missing). A few minutes.
!pip install -q contrastive-lm==0.1.0 vllm==0.19.1 2>&1 | grep -v "dependency conflicts\|incompatible" | tail -5
!pip list 2>/dev/null | grep -E "^(vllm|torch|contrastive-lm|transformers) " | tee {LOG}/versions.txt

In [ ]:
# 3 · Start the Qwen3-8B embedding server (the CLM repo's serve_qwen3_8b.sh settings, whole GPU)
vllm = subprocess.Popen(f"exec vllm serve Qwen/Qwen3-8B --served-model-name qwen3-8b --runner pooling "
                        f"--enforce-eager --enable-prefix-caching --max-model-len 2048 "
                        f"--gpu-memory-utilization 0.90 --max-num-seqs 32 --port 8090 > {LOG}/vllm.log 2>&1",
                        shell=True)
wait_http("http://127.0.0.1:8090/health", timeout=2700, log=f"{LOG}/vllm.log", proc_name="vLLM Qwen3-8B", proc=vllm)

In [ ]:
# 4 · Start clm-serve (downloads the 75 MB reference head) and sanity-check it
clm = subprocess.Popen(f"exec clm-serve --no-ui --port 8700 > {LOG}/clm.log 2>&1", shell=True)
wait_http("http://127.0.0.1:8700/health", timeout=900, log=f"{LOG}/clm.log", proc_name="clm-serve", proc=clm)
import json  # noqa: E402
import urllib.request  # noqa: E402
req = urllib.request.Request("http://127.0.0.1:8700/v1/systemone", headers={"Content-Type": "application/json"},
    data=json.dumps({"state": {"question": "What causes tides on Earth?"}, "questions": {"best": {
        "type": "choice", "instructions": "Which memory helps answer the question?",
        "criteria": {"a": "The Moon's gravitational pull.", "b": "Photosynthesis in plants."}}}}).encode())
print(json.load(urllib.request.urlopen(req, timeout=120))["answers"]["best"])

In [ ]:
# 5 · Judge all pools. rank = CLM's native layout (the question is the instruction, each memory a
#     candidate); choice = the question in the state plus a fixed "which memory" instruction; noul = yes/no.
env = {"CLM_BASE_URL": "http://127.0.0.1:8700", "SMRITI_LAB_JUDGE_CACHE": f"{OUT}/judge-cache.sqlite"}
MODES = ("rank", "choice", "noul")
for mode, split in [(m, s) for m in MODES for s in ("dev", "test")]:
    run(f"python bench/lab/judge_head/pool_eval.py clm:{mode} {POOLS}/pools-{split}.jsonl.gz "
        f"{OUT}/clm-{mode}-{split}.json --workers 8 --probe 30", res, env=env)

In [ ]:
# 6 · Scoreboard
S = {(m, s): json.load(open(f"{OUT}/clm-{m}-{s}.json"))["summary"] for m in MODES for s in ("dev", "test")}
json.dump({"commit": COMMIT, "gpu": name, "gpu_mib": int(mem),
           "versions": open(f"{LOG}/versions.txt").read()}, open(f"{OUT}/env.json", "w"), indent=1)
print(f"{'Judge':34s} {'dev AUC':>8s} {'test AUC':>9s} {'test top-1':>11s} {'ms/question':>12s}")
print(f"{'Smriti own order':34s} {S['rank','dev']['auc_smriti']:8.3f} {S['rank','test']['auc_smriti']:9.3f} "
      f"{100*S['rank','test']['top1_smriti']:10.1f}% {'-':>12s}")
for m in MODES:
    d, t = S[m, "dev"], S[m, "test"]
    print(f"{'CLM-8B ' + m:34s} {d['auc_judge']:8.3f} {t['auc_judge']:9.3f} {100*t['top1_judge']:10.1f}% "
          f"{t['ms_per_question_p50']:12.0f}")
print(f"{'(ref) hosted Jev, round 3':34s} {0.953:8.3f} {0.954:9.3f} {72.5:10.1f}% {1391:12.0f}")
print(f"{'(ref) Laya + Smriti head, round 2':34s} {'':8s} {0.928:9.3f}")
print("errors:", {k: v["errors"] for k, v in S.items()})
res.push()
print("\nDone. Tell Claude the run finished (Drive file id above).")